# Load Inventory Data into Fabric Lakehouse

## Overview
This notebook loads 5 CSV files from the inventory data folder into Delta tables:
- **DemandForecast.csv** → `inventory.DemandForecast` 
- **Inventory.csv** → `inventory.Inventory`
- **InventoryTransactions.csv** → `inventory.InventoryTransactions`
- **PurchaseOrders.csv** → `inventory.PurchaseOrders`
- **PurchaseOrderItems.csv** → `inventory.PurchaseOrderItems`

## Prerequisites
- CSV files uploaded to **Files/data/inventory** in your Fabric lakehouse
- Schema tables created using **model_inventory.ipynb**

## 1. Configuration and Setup

In [ ]:
# Import required libraries
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from datetime import datetime
import os

# Configuration settings
SCHEMA_NAME = "inventory"
DATA_PATH = "Files/data/inventory"  # Fabric lakehouse path

# File mappings
CSV_FILES = {
    "DemandForecast": f"{DATA_PATH}/DemandForecast.csv",
    "Inventory": f"{DATA_PATH}/Inventory.csv", 
    "InventoryTransactions": f"{DATA_PATH}/InventoryTransactions.csv",
    "PurchaseOrders": f"{DATA_PATH}/PurchaseOrders.csv",
    "PurchaseOrderItems": f"{DATA_PATH}/PurchaseOrderItems.csv"
}

print("✅ Configuration complete!")
print(f"Schema: {SCHEMA_NAME}")
print(f"Data Path: {DATA_PATH}")
print(f"Files to load: {len(CSV_FILES)}")

## 2. Load CSV Files into DataFrames

In [ ]:
# Function to load CSV with error handling
def load_csv_to_dataframe(file_path, table_name):
    """Load CSV file into Spark DataFrame with schema inference"""
    try:
        print(f"📁 Loading {table_name} from {file_path}...")
        
        # Read CSV with header and infer schema
        df = spark.read.option("header", "true") \
                      .option("inferSchema", "true") \
                      .option("timestampFormat", "yyyy-MM-dd HH:mm:ss") \
                      .option("dateFormat", "yyyy-MM-dd") \
                      .csv(file_path)
        
        # Get record count and schema info
        row_count = df.count()
        column_count = len(df.columns)
        
        print(f"   ✅ Loaded {row_count:,} rows, {column_count} columns")
        return df, row_count
        
    except Exception as e:
        print(f"   ❌ Error loading {table_name}: {str(e)}")
        return None, 0

# Load all CSV files
dataframes = {}
total_records = 0

for table_name, file_path in CSV_FILES.items():
    df, count = load_csv_to_dataframe(file_path, table_name)
    if df is not None:
        dataframes[table_name] = df
        total_records += count

print(f"\n🎯 Successfully loaded {len(dataframes)}/{len(CSV_FILES)} files")
print(f"📊 Total records across all files: {total_records:,}")

## 3. Data Validation and Preview

In [ ]:
# Preview each loaded DataFrame
for table_name, df in dataframes.items():
    print(f"\n📋 {table_name} DataFrame Schema:")
    df.printSchema()
    
    print(f"\n🔍 Sample {table_name} Data (first 3 rows):")
    df.show(3, truncate=False)
    
    print("-" * 80)

## 4. Load Data into Delta Tables

In [ ]:
# Function to load DataFrame into Delta table
def load_to_delta_table(df, table_name):
    """Load DataFrame into Delta table with error handling"""
    try:
        full_table_name = f"{SCHEMA_NAME}.{table_name}"
        print(f"💾 Loading {table_name} into {full_table_name}...")
        
        # Write DataFrame to Delta table (overwrite mode)
        df.write.format("delta") \
               .mode("overwrite") \
               .option("mergeSchema", "true") \
               .saveAsTable(full_table_name)
        
        # Verify the load
        loaded_count = spark.sql(f"SELECT COUNT(*) as count FROM {full_table_name}").collect()[0]['count']
        print(f"   ✅ Successfully loaded {loaded_count:,} records to {full_table_name}")
        
        return True, loaded_count
        
    except Exception as e:
        print(f"   ❌ Error loading {table_name}: {str(e)}")
        return False, 0

# Load each DataFrame into its corresponding Delta table
successful_loads = 0
total_loaded_records = 0

# Load in logical order (less dependent tables first)
load_order = ["DemandForecast", "Inventory", "InventoryTransactions", "PurchaseOrders", "PurchaseOrderItems"]

for table_name in load_order:
    if table_name in dataframes:
        success, count = load_to_delta_table(dataframes[table_name], table_name)
        if success:
            successful_loads += 1
            total_loaded_records += count

print(f"\n🎯 Loading Summary:")
print(f"   ✅ Successfully loaded: {successful_loads}/{len(dataframes)} tables")
print(f"   📊 Total records loaded: {total_loaded_records:,}")
print(f"   🏁 Load operation completed!")

## 5. Verify Data Load and Table Health

In [ ]:
# Verify all tables exist and have data
print("🔍 Table Verification Report")
print("=" * 50)

verification_queries = {
    "DemandForecast": f"SELECT COUNT(*) as total_forecasts, COUNT(DISTINCT ProductID) as unique_products FROM {SCHEMA_NAME}.DemandForecast",
    "Inventory": f"SELECT COUNT(*) as total_items, SUM(CurrentStock) as total_stock FROM {SCHEMA_NAME}.Inventory", 
    "InventoryTransactions": f"SELECT COUNT(*) as total_transactions, COUNT(DISTINCT ProductID) as products_with_transactions FROM {SCHEMA_NAME}.InventoryTransactions",
    "PurchaseOrders": f"SELECT COUNT(*) as total_orders, COUNT(DISTINCT SupplierID) as unique_suppliers FROM {SCHEMA_NAME}.PurchaseOrders",
    "PurchaseOrderItems": f"SELECT COUNT(*) as total_line_items, SUM(QuantityOrdered) as total_ordered_qty FROM {SCHEMA_NAME}.PurchaseOrderItems"
}

for table_name, query in verification_queries.items():
    try:
        result = spark.sql(query).collect()[0]
        print(f"\n📊 {table_name}:")
        for field_name in result.__fields__:
            value = result[field_name]
            if isinstance(value, (int, float)) and value > 1000:
                print(f"   {field_name}: {value:,}")
            else:
                print(f"   {field_name}: {value}")
                
    except Exception as e:
        print(f"\n❌ {table_name}: Error - {str(e)}")

print("\n" + "=" * 50)

## 6. Sample Data Queries

In [ ]:
# Sample queries to explore the loaded data
print("🔎 Sample Data Exploration")
print("=" * 40)

# 1. Current inventory status by category
print("\n1️⃣ Current Inventory Status by Product Category:")
query1 = f"""
SELECT ProductCategory, 
       COUNT(*) as product_count,
       SUM(CurrentStock) as total_stock,
       SUM(AvailableStock) as available_stock,
       AVG(AverageCost) as avg_cost
FROM {SCHEMA_NAME}.Inventory 
GROUP BY ProductCategory 
ORDER BY total_stock DESC
"""
spark.sql(query1).show()

# 2. Recent purchase orders
print("\n2️⃣ Recent Purchase Orders (Top 5):")
query2 = f"""
SELECT PurchaseOrderNumber, SupplierName, OrderDate, 
       Status, TotalOrderValue, Priority
FROM {SCHEMA_NAME}.PurchaseOrders 
ORDER BY OrderDate DESC 
LIMIT 5
"""
spark.sql(query2).show()

# 3. Inventory transaction summary
print("\n3️⃣ Inventory Transaction Types Summary:")
query3 = f"""
SELECT TransactionType, 
       COUNT(*) as transaction_count,
       SUM(Quantity) as total_quantity,
       AVG(UnitCost) as avg_unit_cost
FROM {SCHEMA_NAME}.InventoryTransactions 
GROUP BY TransactionType 
ORDER BY transaction_count DESC
"""
spark.sql(query3).show()

print("\n🎉 Data load verification completed successfully!")

## ✅ Load Operation Complete!

Your inventory data has been successfully loaded into the Fabric lakehouse. 

### Next Steps:
1. **Explore the data** using the sample queries above
2. **Create reports** and dashboards in Power BI
3. **Run analysis** on inventory levels, purchase patterns, and demand forecasts
4. **Set up alerts** for low stock levels and overdue purchase orders

### Loaded Tables:
- `inventory.DemandForecast` - Predictive demand analytics
- `inventory.Inventory` - Current stock levels and warehouse info
- `inventory.InventoryTransactions` - Complete audit trail of stock movements  
- `inventory.PurchaseOrders` - Purchase order headers
- `inventory.PurchaseOrderItems` - Detailed line items with quantities and costs